<a href="https://www.kaggle.com/code/tokarserhii/dz-2026-01-28?scriptVersionId=294847336" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

Завдання:

=====================================================================================
Ознайомитись з даними та структурою даних
Створіть доповнення(transform) для тренувальних та тестових даних. посилання
Створіть train_dataset та test_dataset за допомогою ImageFolder(папка train)
Переконайтесь що у вас привильні назви класів
Створіть DataLoader для тренувальних та тестових даних
Візуалізуйте дані
Збережіть kaggle notebook для подальшої роботи
 
На основі train_dataset та test_dataset з попереднього завдання створити train_loader та test_loader
Створити нейромережу:
використайте 3-5 шари
перший шар Flatten
кількість нейронів у шарах має не збільшуватись
використайте функції активації RELU або LeakyRELU
Збережіть kaggle notebook для подальшої роботи
 
На основі train_dataset та test_dataset з попереднього завдання створити train_loader та test_loader
Створити згорткову нейромережу:
розмір фільтрів - 3
MaxPooling - kernel_size=2, stride=2
можете змінити розмір зображення в transformer до 64
Виведіть confussion matrix та основні метрики
Збережіть kaggle notebook для подальшої роботи
 

In [ ]:
import torch  
from torchvision import datasets, transforms 

In [ ]:
data_dir = "/kaggle/input/fruit-recognition/train/train"

In [ ]:
all_dataset = datasets.ImageFolder(data_dir)

In [ ]:
all_dataset.classes

In [ ]:
num_classes = len(all_dataset.classes)
num_classes

In [ ]:
from torchvision import transforms

IMG_SIZE = 64  # "можете змінити до 64"

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])



In [ ]:
len(all_dataset)

In [ ]:


data_train, data_test = torch.utils.data.random_split(all_dataset,[0.8, 0.2])


In [ ]:
print(len(data_train), len(data_test), len(data_train) + len(data_test))

In [ ]:
from torch.utils.data import Dataset,DataLoader

class TransformDataset(Dataset):
    def __init__(self, dataset, transformer):
        super().__init__()
        self.dataset = dataset
        self.transformer = transformer

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, y = self.dataset[idx]
        new_img = self.transformer(img)
        return new_img, y

In [ ]:
train_data = TransformDataset(data_train,train_transform)
test_data = TransformDataset(data_test,test_transform)
print(len(train_data), len(test_data))

In [ ]:
train_loader = DataLoader(train_data, batch_size=64)
test_loader = DataLoader(test_data, batch_size=64)



In [ ]:
for img,idx in test_loader:
     print(img.shape)
     print(idx.shape) 

In [ ]:


import matplotlib.pyplot as plt

for i in range(3):  # Show 3 images

    # Get the image data (tensor) and convert it back to a NumPy array for manipulation
    img, y = train_data[i]
    img = img.numpy()
    
    # Convert the color channels from (channels, height, width) to (height, width, channels) for pyplot
    img = img.transpose((1, 2, 0))
    print(img.shape)
    
    # Get the label name from the dataset class labels
    label = all_dataset.classes[y]

    # Plot the image with a title (including label name)
    plt.imshow(img)
    plt.title(f"Label {label}")
    plt.show()



In [ ]:
from torchvision.utils import make_grid
loader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=32)
batch, labels = next(iter(loader))
grid = make_grid(batch).permute(1, 2, 0) # результатом є тензор
plt.imshow(grid)

In [ ]:
batch.shape

In [ ]:
img, label = train_data[0]
img.shape

In [ ]:
for img,idx in test_loader:
    break

img.shape

In [ ]:
device = 'cuda'
16*28*28

In [ ]:

import torch
import torch.nn as nn

# num_classes = len(train_dataset.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3, padding=1),  # filter 3x3
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),      # 2x2, stride 2

    nn.Conv2d(8, 16, kernel_size=3, padding=1), # filter 3x3
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(16, 32, kernel_size=3, padding=1),# filter 3x3
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.AdaptiveAvgPool2d((4,4)),
    nn.Flatten(),
    nn.Linear(32*4*4, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)                 #  33 класи
).to(device)


In [ ]:
for imgs, labels in train_loader:
    break
imgs.shape

In [ ]:
import torch
from tqdm import tqdm

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_idx, (imgs, labels) in enumerate(tqdm(train_loader, leave=False)):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 20 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}")

    print(f"===== Epoch {epoch+1} finished. Avg train loss: {running_loss/len(train_loader):.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix shape:", cm.shape)  # має бути (33, 33)
print(cm)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision (macro):", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Recall (macro):", recall_score(y_true, y_pred, average="macro", zero_division=0))
print("F1 (macro):", f1_score(y_true, y_pred, average="macro", zero_division=0))

print("\nReport:\n", classification_report(
    y_true, y_pred, zero_division=0
))

За результатами тестування модель показала високу якість класифікації. Матриця помилок має майже всі значення на головній діагоналі, що свідчить про коректне розпізнавання класів. Основні метрики (accuracy, precision, recall, F1-score) наближаються до 1.0, що означає високу точність та стабільність моделі для всіх 33 класів.